# Tipado y modelos de datos con Pydantic

**Unidad 1 · Sesión 2**

Anotaciones de tipo, análisis estático y validación de datos en Python.

## Contenido

| Sección | Tema |
|---|---|
| 0 | Preparación |
| 1 | Anotaciones de tipo y colecciones |
| 2 | Análisis estático con mypy |
| Ejemplo | Descuentos y precisión del tipo de retorno |
| 3 | Modelos con Pydantic |
| 4 | Restricciones y errores de validación |
| 5 | Validadores de campos |
| 6 | Configuración |
| Ejemplo | Registro de usuarios |
| 7 | Modelos anidados y serialización |
| 8 | Módulo reutilizable |
| 9 | Procesamiento de registros |
| 10 | Una API pequeña con FastAPI y Pydantic |

## Objetivos

- Anotar funciones y colecciones e interpretar diagnósticos de mypy.
- Definir modelos con restricciones y validadores.
- Localizar errores en datos de entrada.
- Serializar modelos anidados y utilizarlos desde un módulo.
- Observar la validación de una petición a una API.

## 0. Preparación

>**mypy** es un verificador estático: analiza el código y contrasta las operaciones
con sus tipos declarados. **Pydantic** es una biblioteca que utiliza anotaciones
para validar y convertir datos durante la ejecución.

Ejecuta la siguiente celda en Colab o Jupyter. Desde terminal puedes instalar las
mismas dependencias con `python -m pip install -r requirements.txt`.

In [37]:
%pip install -q pydantic==2.12.5 mypy==1.19.1 email-validator==2.3.0 fastapi==0.141.1 httpx2==2.13.1

In [38]:
import sys
from importlib.metadata import version

assert sys.version_info >= (3, 12), "Use Python 3.12 or newer"
print(f"Python: {sys.version.split()[0]}")
print(f"Pydantic: {version('pydantic')}")
print(f"mypy: {version('mypy')}")

Python: 3.13.15
Pydantic: 2.12.5
mypy: 1.19.1


## 1. Del comportamiento al contrato

Una anotación indica el tipo esperado. En `score: float`, `score` es el nombre y
`float` el tipo. Después de los parámetros, `-> str` describe el valor de retorno.
La implementación sigue siendo necesaria: los tipos no calculan el resultado.

**Las anotaciones no validan ni convierten automáticamente los argumentos cuando
Python ejecuta una función.**

In [39]:
def repeat_text(text: str, times: int) -> str:
    return text * times


print(repeat_text("hello ", 2))
# Intentional mismatch: Python executes this even though the annotation says str.
print(repeat_text(3, 2))
assert repeat_text(3, 2) == 6

hello hello 
6


### Análisis estático y ejecución

Las anotaciones describen los tipos esperados de variables, parámetros y retornos.
Mypy utiliza esa información para detectar operaciones incompatibles sin ejecutar
el programa. Las comprobaciones con `assert` verifican resultados concretos cuando
Python sí ejecuta el código.

Estas herramientas tienen alcances diferentes. Una función puede superar el
análisis estático y recibir después un dato externo inválido. El siguiente ejemplo
analiza con mypy la misma llamada que Python aceptó en la celda anterior.

Aquí usamos `mypy_api.run` para analizar texto sin crear un archivo: `-c` pasa
ese texto. `--strict` añade reglas, como exigir funciones anotadas;
`--no-incremental` evita reutilizar resultados de la caché. Más adelante
veremos el comando habitual
sobre un archivo `.py`.

In [40]:
from mypy import api as mypy_api

repeat_source = """def repeat_text(text: str, times: int) -> str:
    return text * times

result = repeat_text(3, 2)
"""
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", repeat_source]
)
print(stdout)
assert exit_status == 1
assert "arg-type" in stdout

runtime_result = repeat_text(3, 2)
print(f"Runtime result: {runtime_result!r}")
assert runtime_result == 6

<string>:4: error: Argument 1 to "repeat_text" has incompatible type "int"; expected "str"  [arg-type]
Found 1 error in 1 file (checked 1 source file)

Runtime result: 6


### Colecciones y valores que pueden faltar

`list[float]` describe una lista de números; `dict[str, int]`, un diccionario de
claves de texto y valores enteros. `float | None` permite un número o ausencia.
No significa «cero». Un argumento con `= None` además puede omitirse en la llamada.

In [41]:
scores: list[float] = [0.0, 0.5, 1.0]
counts: dict[str, int] = {"positive": 2, "neutral": 1}
coordinates: tuple[float, float] = (20.97, -89.62)


def mean_score(values: list[float]) -> float | None:
    if not values:
        return None
    return sum(values) / len(values)


assert mean_score([]) is None
assert mean_score([0.0]) == 0.0
print(mean_score(scores))


def resolve_threshold(threshold: float | None = None) -> float:
    if threshold is None:
        return 0.5
    return threshold


assert resolve_threshold() == 0.5  # The argument is omitted.
assert resolve_threshold(None) == 0.5  # None is passed explicitly.
assert resolve_threshold(0.0) == 0.0  # Zero is a value, not absence.

0.5


### Ejercicio 1 · Anotar sin cambiar el comportamiento

Anota `select_scores`: recibe una lista de floats y un umbral float; devuelve
una lista de floats. Conserva los valores iguales al umbral. Comprueba lista vacía,
umbral cero y `[0.5, 0.9, 1.0]` con umbral 0.9.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [42]:
# Add annotations and checks.
from mypy import api as mypy_api


# Mi predicción: la función se va a comportar igual que antes.
# Las anotaciones solo describen los tipos; no cambian lo que hace.
# Con umbral 0.9 creo que conserva 0.9 y 1.0, porque usa >=.

# Agrego las anotaciones:
# values es una lista de floats, threshold es un float,
# y la función devuelve una lista de floats.
def select_scores(values: list[float], threshold: float) -> list[float]:
    return [value for value in values if value >= threshold]


# Comprobación 1: lista vacía. No hay nada que seleccionar.
assert select_scores([], 0.9) == []

# Comprobación 2: umbral cero. Se conservan todos los valores mayores o iguales a 0,
# incluido el 0.0, porque cero cumple 0.0 >= 0.0.
assert select_scores([0.0, 0.5], 0.0) == [0.0, 0.5]

# Comprobación 3: con umbral 0.9 se conserva el 0.9 porque es igual al umbral.
assert select_scores([0.5, 0.9, 1.0], 0.9) == [0.9, 1.0]
print(select_scores([0.5, 0.9, 1.0], 0.9))

# Comprobación extra: le pido a mypy que revise la función anotada.
codigo = '''def select_scores(values: list[float], threshold: float) -> list[float]:
    return [value for value in values if value >= threshold]

resultado = select_scores([0.5, 0.9, 1.0], 0.9)
'''
stdout, stderr, exit_status = mypy_api.run(["--strict", "--no-incremental", "-c", codigo])
print(stdout)
assert exit_status == 0

# Explicación:
# Las anotaciones no cambiaron el comportamiento: la función sigue filtrando igual.
# Lo que agregan es un contrato que dice qué recibe y qué devuelve,
# y mypy puede revisarlo sin ejecutar el programa.
# Los valores iguales al umbral se conservan porque la condición es >= y no >.
# Con umbral cero se conserva el 0.0, porque cero es un valor real, no ausencia.

[0.9, 1.0]
Success: no issues found in 1 source file



## 2. Leer un error del verificador

Esta celda analiza otra cadena de código sin ejecutarla ni crear archivos.
`mypy_api.run` devuelve salida, errores internos y código de salida. El código
1 es esperado aquí: hemos
introducido dos errores deliberados.

Lee el mensaje en este orden: ubicación, operación, tipo recibido y tipo esperado.
No cambies un tipo para callar al verificador: revisa qué comportamiento necesitas.

In [43]:
broken_source = """def apply_threshold(score: float, threshold: float) -> bool:
    return score >= threshold

result = apply_threshold("0.9", 0.8)

def display_score(score: float | None) -> str:
    return f"{score * 100:.1f}%"
"""
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", broken_source]
)
print(stdout)
assert exit_status == 1
assert "arg-type" in stdout
assert "operator" in stdout

<string>:4: error: Argument 1 to "apply_threshold" has incompatible type "str"; expected "float"  [arg-type]
<string>:7: error: Unsupported operand types for * ("None" and "int")  [operator]
<string>:7: note: Left operand is of type "float | None"
Found 2 errors in 1 file (checked 1 source file)



### Ejercicio 2 · Corregir el contrato

Corrige los dos problemas de `broken_source`: convierte explícitamente el texto
a float y atiende `None` antes de multiplicar. Trabaja en otra cadena para conservar
el ejemplo defectuoso. Ejecuta mypy y comprueba código de salida 0. ¿Por qué convertir
con `float` todavía puede fallar en tiempo de ejecución?

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [44]:
# Write fixed_source and run mypy_api.run on it.

# Mi predicción: si convierto el texto con float() y reviso None
# antes de multiplicar, mypy ya no va a encontrar errores.

# Escribo el código corregido en una cadena nueva (fixed_source),
# para no modificar broken_source y conservar el ejemplo con errores.
fixed_source = """def apply_threshold(score: float, threshold: float) -> bool:
    return score >= threshold

result = apply_threshold(float("0.9"), 0.8)

def display_score(score: float | None) -> str:
    if score is None:
        return "Sin dato"
    return f"{score * 100:.1f}%"
"""
# Corrección 1: convierto "0.9" a número con float() antes de pasarlo,
# porque apply_threshold espera un float y no un texto.
# Corrección 2: en display_score reviso si score es None antes de multiplicar.
# Si no hay dato, devuelvo "Sin dato" en lugar de intentar multiplicar None.

# Le pido a mypy que revise el código corregido.
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", fixed_source]
)
print(stdout)

# Código de salida 0 significa que mypy no encontró errores.
assert exit_status == 0

# Ahora ejecuto el código corregido para comprobar que también funciona con datos reales.
# exec() ejecuta el texto y guarda lo que crea en el diccionario espacio.
espacio: dict[str, object] = {}
exec(fixed_source, espacio)

# apply_threshold(0.9, 0.8) debe dar True, porque 0.9 es mayor o igual a 0.8.
assert espacio["result"] is True

# Pruebo display_score con un valor, con cero y con None.
display_score = espacio["display_score"]
print(display_score(0.875))
print(display_score(0.0))
print(display_score(None))
assert display_score(0.875) == "87.5%"
assert display_score(0.0) == "0.0%"      # el cero se muestra como dato
assert display_score(None) == "Sin dato" # None se muestra como ausencia

# Pregunta: ¿por qué float() todavía puede fallar al ejecutar?
# mypy solo revisa que le pase un texto a float(), y eso está permitido.
# Pero no puede saber si ese texto tiene un número válido.
# Si llega un texto como "invalid", float() falla con ValueError al ejecutar:
try:
    float("invalid")
except ValueError as error:
    print("Error en ejecución:", error)

# Explicación:
# mypy revisa los tipos sin ejecutar el código, pero no revisa el contenido
# de los datos que llegan después. Por eso un programa puede pasar mypy
# y aun así fallar con datos reales mal capturados.
# Es como revisar que un formato tenga el campo de pH como numérico:
# eso no garantiza que alguien no escriba "siete" al llenarlo en campo.
# Para validar los datos reales al ejecutar se usa Pydantic, que viene a continuación.

Success: no issues found in 1 source file

87.5%
0.0%
Sin dato
Error en ejecución: could not convert string to float: 'invalid'


### Precisión del tipo de retorno

El tipo de retorno determina qué operaciones puede comprobar el verificador.
`Iterable[float]` garantiza que los valores pueden recorrerse, pero no garantiza
`len()` ni acceso por índice.

La función siguiente devuelve una lista y declara `list[float]` como retorno.
`Iterable` describe la entrada sin exigir que sea una lista. Su funcionamiento
se estudiará con detalle en la próxima sesión.

In [45]:
from collections.abc import Iterable


def calculate_discounts(items: Iterable[float], discount: float) -> list[float]:
    return [item * (1 - discount) for item in items]


prices = [100.0, 200.0, 300.0]
discounted_prices = calculate_discounts(prices, 0.2)
print(len(discounted_prices))
print(discounted_prices[0])
assert discounted_prices == [80.0, 160.0, 240.0]

3
80.0


Ahora declaramos `Iterable[float]` como retorno, aunque la implementación todavía
construye una lista. Python permite `len()` y el índice porque observa el objeto real;
mypy los rechaza porque solo puede garantizar lo que declara el contrato.

In [46]:
iterable_return_source = """from collections.abc import Iterable

def calculate_discounts(
    items: Iterable[float], discount: float
) -> Iterable[float]:
    return [item * (1 - discount) for item in items]

discounted_prices = calculate_discounts([100.0, 200.0], 0.2)
count = len(discounted_prices)
first = discounted_prices[0]
"""
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", iterable_return_source]
)
print(stdout)
assert exit_status == 1

runtime_namespace: dict[str, object] = {}
exec(iterable_return_source, runtime_namespace)
assert runtime_namespace["count"] == 2
assert runtime_namespace["first"] == 80.0

<string>:9: error: Argument 1 to "len" has incompatible type "Iterable[float]"; expected "Sized"  [arg-type]
<string>:10: error: Value of type "Iterable[float]" is not indexable  [index]
Found 2 errors in 1 file (checked 1 source file)



### Evitar `Any` como respuesta automática

`Any` permite saltarse muchas comprobaciones. Si un dato crudo puede ser cualquier
objeto, `object` obliga a comprobar su tipo antes de usar operaciones específicas.
Con `isinstance` podemos acotar el tipo. En los modelos, `Literal` limitará los
valores permitidos. Hoy no necesitamos genéricos avanzados ni protocolos.

In [47]:
def describe_input(value: object) -> str:
    if isinstance(value, str):
        return f"Text with {len(value)} characters"
    return "Not a text value"


print(describe_input("hello"))
print(describe_input(None))

Text with 5 characters
Not a text value


## 3. Un modelo de datos también es una clase

`BaseModel` es la clase base de Pydantic. Al heredar de ella y declarar campos,
obtenemos construcción, validación y serialización. Una instancia sigue teniendo
atributos: `prediction.score`, en lugar de `record["score"]`.

Comenzamos con un modelo deliberadamente pequeño para observar sus límites.

In [48]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator


class BasicPrediction(BaseModel):
    text: str
    score: float


basic_prediction = BasicPrediction.model_validate({"text": "Good service", "score": "0.9"})
print(basic_prediction)
print(type(basic_prediction.score))
assert basic_prediction.score == 0.9
# A float annotation alone does not constrain the numerical range.
print(BasicPrediction(text="Outside the range", score=1.5))

text='Good service' score=0.9
<class 'float'>
text='Outside the range' score=1.5


### Conversión y modo estricto

En modo habitual, Pydantic acepta algunas conversiones, como `"0.9"` a `0.9`.
**Convertir** no es lo mismo que **rechazar todo valor cuyo tipo original difiera**.
El modo estricto restringe conversiones; las restricciones de rango son otra regla.

Decidiremos el contrato según la fuente: un archivo puede contener números como
texto, mientras que una configuración interna puede exigir enteros reales.

In [49]:
try:
    BasicPrediction.model_validate({"text": "Good service", "score": "0.9"}, strict=True)
except ValidationError as error:
    print(error.errors()[0]["type"])
else:
    raise AssertionError("Strict validation should reject the string score")

float_type


## 4. Campos restringidos y errores útiles

`Literal` enumera etiquetas válidas; `Field` añade límites. `Annotated` combina
un tipo con metadatos: el alias `Score` se puede reutilizar sin repetir sus reglas.
`source: str | None = None` permite ausencia y también omitir el campo.

`extra="forbid"` rechaza claves desconocidas, como `scroe`; de otro modo un error
de escritura podría pasar inadvertido. Este primer modelo aún no normaliza texto.

In [50]:
from typing import Annotated, Literal

Label = Literal["positive", "neutral", "negative"]
Score = Annotated[float, Field(ge=0, le=1, allow_inf_nan=False)]


class ConstrainedPrediction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None


constrained = ConstrainedPrediction(text="Good", label="positive", score=0.0)
assert constrained.source is None
print(constrained)

text='Good' label='positive' score=0.0 source=None


In [51]:
invalid_record = {"text": "", "label": "unknown", "score": 1.2, "scroe": 0.8}
try:
    ConstrainedPrediction.model_validate(invalid_record)
except ValidationError as error:
    for detail in error.errors():
        print(detail["loc"], detail["type"], detail["msg"])
else:
    raise AssertionError("The record should be rejected")

('text',) string_too_short String should have at least 1 character
('label',) literal_error Input should be 'positive', 'neutral' or 'negative'
('score',) less_than_equal Input should be less than or equal to 1
('scroe',) extra_forbidden Extra inputs are not permitted


Cada entrada de `errors()` describe un problema: `loc` indica el campo, `type`
su categoría y `msg` una explicación. Un registro puede producir varios errores.
La función que procesa un lote debe conservarlos, sin confundir número de errores
con número de registros rechazados.

### Ejercicio 3 · Ausente, nulo e inválido

Prueba registros que omitan `source`, que usen `source=None`, que omitan `score`
y que usen `score=None`. Predice cuáles se aceptan. Comprueba también límites 0 y 1.
Explica por qué `str | None` sin `= None` permitiría nulo pero seguiría siendo obligatorio.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [52]:
# Build each input and catch only ValidationError for expected failures.

# Mis predicciones:
# 1. Omitir source: se acepta, porque source tiene "= None" como valor por defecto.
# 2. source=None: se acepta, porque el tipo permite None (str | None).
# 3. Omitir score: se rechaza, porque score es obligatorio.
# 4. score=None: se rechaza, porque score debe ser un número.
# 5. score en 0 y en 1: se aceptan, porque los límites ge=0 y le=1 incluyen los extremos.

# Caso 1: omito source.
sin_source = ConstrainedPrediction.model_validate(
    {"text": "Buen servicio", "label": "positive", "score": 0.9}
)
print("1. Sin source:", sin_source)
assert sin_source.source is None

# Caso 2: source=None de forma explícita.
source_nulo = ConstrainedPrediction.model_validate(
    {"text": "Buen servicio", "label": "positive", "score": 0.9, "source": None}
)
print("2. source=None:", source_nulo)
assert source_nulo.source is None

# Caso 3: omito score. Debe fallar con un error de tipo "missing" (falta).
# Solo atrapo ValidationError, que es el error esperado.
try:
    ConstrainedPrediction.model_validate({"text": "Buen servicio", "label": "positive"})
except ValidationError as error:
    detalle = error.errors()[0]
    print("3. Sin score:", detalle["loc"], detalle["type"])
    assert detalle["type"] == "missing"
else:
    raise AssertionError("El registro sin score debería rechazarse")

# Caso 4: score=None. Debe fallar porque None no es un número.
try:
    ConstrainedPrediction.model_validate(
        {"text": "Buen servicio", "label": "positive", "score": None}
    )
except ValidationError as error:
    detalle = error.errors()[0]
    print("4. score=None:", detalle["loc"], detalle["type"])
    assert detalle["type"] == "float_type"
else:
    raise AssertionError("El registro con score=None debería rechazarse")

# Caso 5: límites. 0 y 1 deben aceptarse porque ge significa "mayor o igual"
# y le significa "menor o igual".
limite_cero = ConstrainedPrediction(text="Regular", label="neutral", score=0)
limite_uno = ConstrainedPrediction(text="Excelente", label="positive", score=1)
print("5. Límites aceptados:", limite_cero.score, limite_uno.score)
assert limite_cero.score == 0.0
assert limite_uno.score == 1.0

# Caso 6: justo fuera de los límites, deben rechazarse.
for valor_fuera in (-0.1, 1.1):
    try:
        ConstrainedPrediction(text="Fuera", label="neutral", score=valor_fuera)
    except ValidationError as error:
        print("6. Fuera de rango:", valor_fuera, error.errors()[0]["type"])
    else:
        raise AssertionError(f"{valor_fuera} debería rechazarse")

# Caso 7: demuestro la pregunta del ejercicio con un modelo pequeño.
# Aquí source es str | None pero SIN "= None".
class SourceObligatorio(BaseModel):
    source: str | None

# Acepta None si lo envío de forma explícita...
print("7. Con source=None:", SourceObligatorio(source=None))

# ...pero rechaza el registro si omito el campo.
try:
    SourceObligatorio.model_validate({})
except ValidationError as error:
    print("8. Sin source:", error.errors()[0]["type"])
    assert error.errors()[0]["type"] == "missing"
else:
    raise AssertionError("Sin '= None', el campo debería ser obligatorio")

# Explicación:
# Ausente (no enviar el campo), nulo (enviar None) e inválido (enviar un valor que
# no cumple las reglas) son tres situaciones distintas.
# "str | None" solo dice qué valores se permiten: texto o None.
# "= None" dice algo diferente: que el campo se puede omitir y tomará None por defecto.
# Por eso "str | None" sin "= None" acepta None, pero sigue siendo obligatorio:
# hay que enviar el campo, aunque sea con valor None.
# En un formato de campo sería la diferencia entre "Observaciones: N/A",
# escrito a propósito, y dejar la casilla sin llenar, que obliga a regresar a preguntar.

1. Sin source: text='Buen servicio' label='positive' score=0.9 source=None
2. source=None: text='Buen servicio' label='positive' score=0.9 source=None
3. Sin score: ('score',) missing
4. score=None: ('score',) float_type
5. Límites aceptados: 0.0 1.0
6. Fuera de rango: -0.1 greater_than_equal
6. Fuera de rango: 1.1 less_than_equal
7. Con source=None: source=None
8. Sin source: missing


## 5. Validadores: reglas que necesitan código

Queremos aceptar `" POSITIVE "` y rechazar textos de solo espacios. Antes de validar
los campos, quitaremos espacios externos y convertiremos las cadenas a minúsculas.
Es una decisión explícita de este taller, no una limpieza universal para todos los textos.

`@field_validator(..., mode="before")` registra una función que recibe la entrada
cruda. `@classmethod` indica que recibe la clase como `cls`, no una instancia como
`self`. Los decoradores conectan nuestra función con Pydantic; no necesitas crear
un decorador propio. Devuelve el valor transformado o lanza `ValueError`.

Los booleanos tienen un comportamiento numérico en Python; queremos rechazarlos
explícitamente como scores. La validación de rango rechazará también `NaN` e infinito.

In [53]:
class Prediction(BaseModel):
    """Normalize text fields and validate one incoming prediction."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None

    @field_validator("text", "label", "source", mode="before")
    @classmethod
    def normalize_strings(cls, value: object) -> object:
        if isinstance(value, str):
            return value.strip().lower()
        return value

    @field_validator("score", mode="before")
    @classmethod
    def reject_boolean_score(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a score")
        return value

In [54]:
normalized = Prediction.model_validate(
    {"text": "  GOOD SERVICE  ", "label": " POSITIVE ", "score": "0.9"}
)
assert normalized.text == "good service"
assert normalized.label == "positive"
print(normalized)

text='good service' label='positive' score=0.9 source=None


### Ejercicio 4 · Detectar los casos que faltaban

Prueba texto `"   "`, etiqueta `None`, score `True`, `"nan"`, `-0.1` y `1.1`.
Deben rechazarse. Comprueba también que `0.0` sea válido y que la entrada original
no se modifique al normalizarla. ¿Por qué `min_length=1` sin quitar espacios era insuficiente?

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [55]:

# Validate one changed field at a time and inspect the location of each error.

# Mi predicción: los seis casos se rechazan.
# El texto "   " se rechaza porque al quitar los espacios queda vacío.
# El score 0.0 se acepta, porque cero es un valor válido dentro del rango.

# Parto de un registro válido y cambio un solo campo a la vez,
# así sé exactamente qué campo causó cada error.
registro_base = {"text": "  Buen Servicio  ", "label": " POSITIVE ", "score": 0.9}

# Guardo una copia para comprobar al final que el original no cambió.
copia_original = dict(registro_base)

# Lista de cambios a probar: (campo, valor inválido).
cambios = [
    ("text", "   "),   # solo espacios
    ("label", None),   # etiqueta nula
    ("score", True),   # booleano, no es un score
    ("score", "nan"),  # "not a number", no es un número finito
    ("score", -0.1),   # por debajo de 0
    ("score", 1.1),    # por encima de 1
]

for campo, valor in cambios:
    # Creo un registro nuevo con todo igual, excepto el campo que quiero probar.
    registro = {**registro_base, campo: valor}
    try:
        Prediction.model_validate(registro)
    except ValidationError as error:
        detalle = error.errors()[0]
        print(f"{campo}={valor!r}: rechazado en {detalle['loc']} ({detalle['type']})")
        # Compruebo que el error se ubicó en el campo que cambié.
        assert detalle["loc"] == (campo,)
    else:
        raise AssertionError(f"{campo}={valor!r} debería rechazarse")

# Compruebo que el cero sí es válido.
cero = Prediction.model_validate({**registro_base, "score": 0.0})
print("score=0.0 aceptado:", cero.score)
assert cero.score == 0.0

# Compruebo que normalizar no modifica el diccionario original.
normalizada = Prediction.model_validate(registro_base)
print("Normalizada:", normalizada)
print("Original:   ", registro_base)
assert normalizada.text == "buen servicio"
assert registro_base == copia_original
assert registro_base["text"] == "  Buen Servicio  "

# Explicación:
# min_length=1 solo cuenta caracteres, y los espacios cuentan como caracteres.
# Así, "   " tiene 3 caracteres y pasaría la regla aunque no tenga contenido real.
# Por eso el validador primero quita los espacios con strip(): "   " queda como ""
# y entonces min_length=1 sí lo rechaza.
# En un formato de campo sería como aceptar un campo de "responsable" lleno
# solo de espacios: parece lleno, pero no dice quién es.
# Además, Pydantic crea un objeto nuevo con los valores normalizados
# y no toca el diccionario original, así conservo el dato tal como llegó.

text='   ': rechazado en ('text',) (string_too_short)
label=None: rechazado en ('label',) (literal_error)
score=True: rechazado en ('score',) (value_error)
score='nan': rechazado en ('score',) (finite_number)
score=-0.1: rechazado en ('score',) (greater_than_equal)
score=1.1: rechazado en ('score',) (less_than_equal)
score=0.0 aceptado: 0.0
Normalizada: text='buen servicio' label='positive' score=0.9 source=None
Original:    {'text': '  Buen Servicio  ', 'label': ' POSITIVE ', 'score': 0.9}


### Validar asignaciones tiene un alcance concreto

`validate_assignment=True` vuelve a validar al asignar atributos del modelo.
Sin esa opción, construir una instancia válida no garantiza que una asignación
posterior conserve el contrato. Esta opción no vigila cada modificación interna
de una lista: mantenemos las operaciones del lote en funciones conocidas.

In [56]:
try:
    normalized.score = 2.0
except ValidationError as error:
    print(error.errors()[0]["msg"])
assert normalized.score == 0.9

Input should be less than or equal to 1


## 6. Configuración explícita

La configuración también es dato: nombre del lote, umbral de selección y máximo
de registros. Aquí llega como diccionario local, sin variables de entorno ni secretos.

Distingue `model_config` (comportamiento de Pydantic) de `BatchConfig` (parámetros
de nuestra aplicación). El máximo será un entero estricto entre 1 y 1000: no
aceptaremos `"10"` ni `True`. El umbral comparte el contrato de los scores.

In [57]:
class BatchConfig(BaseModel):
    """Application settings supplied explicitly, without secrets or services."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    batch_name: str = Field(min_length=1)
    threshold: Score = 0.9
    max_records: int = Field(default=100, ge=1, le=1000, strict=True)

    @field_validator("batch_name")
    @classmethod
    def require_nonblank_name(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Batch name must not be blank")
        return value

    @field_validator("threshold", mode="before")
    @classmethod
    def reject_boolean_threshold(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a threshold")
        return value

In [58]:
config = BatchConfig.model_validate({"batch_name": " Demo batch ", "threshold": "0.9"})
assert config.batch_name == "Demo batch"
assert config.max_records == 100
print(config)

batch_name='Demo batch' threshold=0.9 max_records=100


### Ejercicio 5 · Configuración antes de procesar

Construye una configuración con nombre `"Practice"`, umbral 0 y máximo 3.
Después comprueba que máximo `"3"`, máximo 0 y nombre `"   "` produzcan errores.
Explica por qué un error de configuración debe detener el lote completo, mientras
que un registro inválido puede conservarse como rechazo.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [59]:
# Create a valid configuration and try the invalid variants.

# Mi predicción:
# La configuración con nombre "Practice", umbral 0 y máximo 3 es válida.
# max_records="3" se rechaza porque max_records es estricto y no acepta texto.
# max_records=0 se rechaza porque el mínimo es 1.
# Un nombre de solo espacios se rechaza porque el validador lo deja vacío.

# Creo la configuración válida.
config_practica = BatchConfig(batch_name="Practice", threshold=0, max_records=3)
print("Configuración válida:", config_practica)
assert config_practica.batch_name == "Practice"
assert config_practica.threshold == 0.0  # el umbral 0 es válido
assert config_practica.max_records == 3

# Pruebo las tres variantes inválidas, cambiando un solo campo a la vez.
config_base = {"batch_name": "Practice", "threshold": 0, "max_records": 3}
variantes_invalidas = [
    ("max_records", "3", "int_type"),             # texto en vez de entero
    ("max_records", 0, "greater_than_equal"),     # menor que el mínimo de 1
    ("batch_name", "   ", "value_error"),         # nombre en blanco
]

for campo, valor, tipo_esperado in variantes_invalidas:
    try:
        BatchConfig.model_validate({**config_base, campo: valor})
    except ValidationError as error:
        detalle = error.errors()[0]
        print(f"{campo}={valor!r}: rechazado en {detalle['loc']} ({detalle['type']})")
        assert detalle["loc"] == (campo,)
        assert detalle["type"] == tipo_esperado
    else:
        raise AssertionError(f"{campo}={valor!r} debería rechazarse")

# Explicación:
# Un error de configuración debe detener todo el lote porque la configuración
# define cómo se procesan TODOS los registros. Si el umbral o el máximo están mal,
# cada resultado del lote sería incorrecto, aunque cada registro esté bien.
# En cambio, un registro inválido solo afecta a ese registro: se puede rechazar,
# guardar con su error y seguir con los demás.
# En mi trabajo: si el método de muestreo está mal definido (por ejemplo,
# un límite de detección equivocado), hay que detener toda la campaña,
# porque todos los resultados quedarían comprometidos. Pero si una sola muestra
# llegó con el sello roto, se rechaza esa muestra y las demás se analizan normal.

Configuración válida: batch_name='Practice' threshold=0.0 max_records=3
max_records='3': rechazado en ('max_records',) (int_type)
max_records=0: rechazado en ('max_records',) (greater_than_equal)
batch_name='   ': rechazado en ('batch_name',) (value_error)


### Registro de usuarios

El modelo usa `EmailStr` para validar el formato del correo y `SecretStr` para
ocultar la contraseña en su representación habitual. `exclude=True` la excluye
de la serialización. `SecretStr` no cifra ni almacena de forma segura una contraseña.

El rol tiene un valor predeterminado válido. El ejemplo muestra la construcción
del modelo, sus errores de validación y su serialización.

In [60]:
from pydantic import EmailStr, SecretStr


class User(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str = Field(min_length=2)
    email: EmailStr
    password: SecretStr = Field(exclude=True)
    role: Literal["author", "editor", "admin"] = "author"


user_data = {
    "name": "Alex",
    "email": "alex@example.com",
    "password": "ExamplePassword123",
}
user = User.model_validate(user_data)
print(user)
print(user.model_dump_json(indent=2))
assert user.role == "author"
assert "password" not in user.model_dump()

try:
    User.model_validate({**user_data, "email": "invalid-email"})
except ValidationError as error:
    print(error.errors()[0]["loc"], error.errors()[0]["msg"])
else:
    raise AssertionError("Invalid email was accepted")

name='Alex' email='alex@example.com' password=SecretStr('**********') role='author'
{
  "name": "Alex",
  "email": "alex@example.com",
  "role": "author"
}
('email',) value is not a valid email address: An email address must have an @-sign.


Un valor fuera del `Literal` y un campo ausente fallan por motivos distintos.
Comparamos `loc` y `type` para distinguir un valor inválido de un dato requerido.

In [61]:
try:
    User.model_validate({**user_data, "role": "developer"})
except ValidationError as error:
    invalid_role = error.errors()[0]
    print(invalid_role["loc"], invalid_role["type"], invalid_role["msg"])
else:
    raise AssertionError("Invalid role was accepted")

data_without_name = {key: value for key, value in user_data.items() if key != "name"}
try:
    User.model_validate(data_without_name)
except ValidationError as error:
    missing_name = error.errors()[0]
    print(missing_name["loc"], missing_name["type"], missing_name["msg"])
else:
    raise AssertionError("Missing name was accepted")

assert invalid_role["loc"] == ("role",)
assert invalid_role["type"] == "literal_error"
assert missing_name["loc"] == ("name",)
assert missing_name["type"] == "missing"

('role',) literal_error Input should be 'author', 'editor' or 'admin'
('name',) missing Field required


## 7. Modelos anidados y serialización

Un reporte contiene configuración, predicciones y errores: cada componente tiene
su propio modelo. `Field(default_factory=list)` crea una lista por instancia.
Una lista de modelos conserva estructura; no necesitamos diccionarios sin contrato
para pasar resultados entre funciones.

Serializar transforma la representación: `model_dump()` produce un diccionario y
`model_dump_json()` texto JSON. `model_validate_json()` reconstruye y valida desde JSON.
No equivale a escribir automáticamente un archivo ni a guardar en una base de datos.

In [62]:
class ValidationIssue(BaseModel):
    position: int
    field: str
    message: str
    error_type: str


class BatchReport(BaseModel):
    config: BatchConfig
    accepted: list[Prediction] = Field(default_factory=list)
    issues: list[ValidationIssue] = Field(default_factory=list)

In [63]:
first_empty_report = BatchReport(config=config)
second_empty_report = BatchReport(config=config)
first_empty_report.accepted.append(normalized)
assert second_empty_report.accepted == []

example_report = BatchReport(config=config, accepted=[normalized])
payload = example_report.model_dump()
json_text = example_report.model_dump_json(indent=2)
print(json_text)
assert isinstance(payload["accepted"][0], dict)
restored_report = BatchReport.model_validate_json(json_text)
assert restored_report == example_report

{
  "config": {
    "batch_name": "Demo batch",
    "threshold": 0.9,
    "max_records": 100
  },
  "accepted": [
    {
      "text": "good service",
      "label": "positive",
      "score": 0.9,
      "source": null
    }
  ],
  "issues": []
}


### Ejercicio 6 · Ida y vuelta

Crea un reporte vacío, serialízalo a JSON y reconstrúyelo. Comprueba igualdad
y listas vacías. Después cambia el score a 5 en un diccionario obtenido con
`example_report.model_dump()`: al validarlo debe aparecer la ubicación
`("accepted", 0, "score")`. El modelo original debe conservar score 0.9.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [64]:
# Inspect a nested error and check that the source model is unchanged.

# Mi predicción:
# El reporte vacío se convierte a JSON y al reconstruirlo queda idéntico.
# Si cambio el score a 5 en el diccionario, la validación falla porque 5 es mayor que 1,
# y el error indica la ruta completa: lista accepted, posición 0, campo score.
# El modelo original no cambia, porque modifiqué una copia en forma de diccionario.

# Parte 1: ida y vuelta con un reporte vacío.
reporte_vacio = BatchReport(config=config)

# Ida: convierto el reporte a texto JSON.
json_vacio = reporte_vacio.model_dump_json()
print("JSON del reporte vacío:", json_vacio)

# Vuelta: reconstruyo el reporte desde el JSON (y Pydantic lo valida otra vez).
reconstruido = BatchReport.model_validate_json(json_vacio)

# Compruebo que quedó igual y que las listas siguen vacías.
assert reconstruido == reporte_vacio
assert reconstruido.accepted == []
assert reconstruido.issues == []
print("Reconstruido igual al original:", reconstruido == reporte_vacio)

# Parte 2: un error dentro de un modelo anidado.
# model_dump() me da un diccionario independiente del modelo.
datos = example_report.model_dump()

# Cambio el score de la primera predicción aceptada a 5 (fuera de rango).
datos["accepted"][0]["score"] = 5

try:
    BatchReport.model_validate(datos)
except ValidationError as error:
    detalle = error.errors()[0]
    print("Error en:", detalle["loc"], detalle["type"])
    # La ubicación muestra la ruta completa hasta el campo con error.
    assert detalle["loc"] == ("accepted", 0, "score")
else:
    raise AssertionError("El score 5 debería rechazarse")

# Compruebo que el modelo original no cambió.
print("Score original:", example_report.accepted[0].score)
assert example_report.accepted[0].score == 0.9

# Explicación:
# La ubicación ("accepted", 0, "score") se lee como una ruta:
# en la lista accepted, el elemento en la posición 0, el campo score.
# Así sé exactamente dónde está el problema aunque el reporte tenga muchos niveles.
# El modelo original conserva 0.9 porque model_dump() crea un diccionario nuevo;
# lo que cambio en ese diccionario no afecta al modelo.
# En mi trabajo es como un expediente con varias carpetas: el error no dice solo
# "hay un dato mal", sino "carpeta de resultados, muestra 1, campo concentración".

JSON del reporte vacío: {"config":{"batch_name":"Demo batch","threshold":0.9,"max_records":100},"accepted":[],"issues":[]}
Reconstruido igual al original: True
Error en: ('accepted', 0, 'score') less_than_equal
Score original: 0.9


## 8. De la notebook a un módulo reutilizable

La siguiente celda reúne las clases anteriores y
las funciones de procesamiento en `lesson_models.py`, el mismo archivo que acompaña
esta notebook en el repositorio. Así Colab funciona sin descargar archivos auxiliares.

**La celda escribe o reemplaza `lesson_models.py` en el directorio de trabajo.**
Si hiciste cambios propios en ese archivo, guárdalos con otro nombre antes de
volver a ejecutarla. `%%writefile` es una instrucción de Jupyter, no sintaxis Python.

Lee primero `process_records`: valida cada entrada, guarda cada problema con su
posición desde cero y no captura excepciones ajenas a validación. Si el lote supera
`max_records`, lanza `ValueError` antes de procesarlo. El umbral selecciona entre
registros válidos; no convierte en inválidas las predicciones con menor score.

Importamos el módulo como `models`: el prefijo distingue sus clases de las
versiones construidas durante la explicación y evita mezclar sus instancias.

In [65]:
%%writefile lesson_models.py
"""Typed contracts for synthetic prediction records; no model is trained here."""

from typing import Annotated, Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

Label = Literal["positive", "neutral", "negative"]
Score = Annotated[float, Field(ge=0, le=1, allow_inf_nan=False)]


class Prediction(BaseModel):
    """Normalize text fields and validate one incoming prediction."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None

    @field_validator("text", "label", "source", mode="before")
    @classmethod
    def normalize_strings(cls, value: object) -> object:
        if isinstance(value, str):
            return value.strip().lower()
        return value

    @field_validator("score", mode="before")
    @classmethod
    def reject_boolean_score(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a score")
        return value


class BatchConfig(BaseModel):
    """Application settings supplied explicitly, without secrets or services."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    batch_name: str = Field(min_length=1)
    threshold: Score = 0.9
    max_records: int = Field(default=100, ge=1, le=1000, strict=True)

    @field_validator("batch_name")
    @classmethod
    def require_nonblank_name(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Batch name must not be blank")
        return value

    @field_validator("threshold", mode="before")
    @classmethod
    def reject_boolean_threshold(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a threshold")
        return value


class ValidationIssue(BaseModel):
    position: int
    field: str
    message: str
    error_type: str


class BatchReport(BaseModel):
    config: BatchConfig
    accepted: list[Prediction] = Field(default_factory=list)
    issues: list[ValidationIssue] = Field(default_factory=list)


def average_score(predictions: list[Prediction]) -> float | None:
    if not predictions:
        return None
    return sum(prediction.score for prediction in predictions) / len(predictions)


def process_records(records: list[object], config: BatchConfig) -> BatchReport:
    """Reject an oversized batch; preserve all validation issues for each row."""
    if len(records) > config.max_records:
        raise ValueError("Batch exceeds max_records")

    report = BatchReport(config=config)
    for position, record in enumerate(records):
        try:
            prediction = Prediction.model_validate(record)
        except ValidationError as error:
            for detail in error.errors():
                report.issues.append(
                    ValidationIssue(
                        position=position,
                        field=".".join(str(part) for part in detail["loc"]) or "<record>",
                        message=detail["msg"],
                        error_type=detail["type"],
                    )
                )
        else:
            report.accepted.append(prediction)
    return report


def select_predictions(report: BatchReport) -> list[Prediction]:
    """Select by confidence; this is not an accuracy measurement."""
    return [
        prediction
        for prediction in report.accepted
        if prediction.score >= report.config.threshold
    ]

Overwriting lesson_models.py


### Comprobar un archivo desde la terminal

Con el módulo ya guardado, el uso habitual es `mypy --strict lesson_models.py`.
El argumento final es el archivo que mypy revisará. No hace falta `-c` porque
el código ya está en un archivo, ni `--no-incremental` para el trabajo cotidiano.

La siguiente celda ejecuta el equivalente mediante el Python de esta notebook
y comprueba que termina sin errores.

In [66]:
import importlib
import subprocess
import lesson_models as models

# Reload after running the file-writing cell again.
models = importlib.reload(models)
from lesson_models import average_score, process_records, select_predictions

result = subprocess.run(
    [sys.executable, "-m", "mypy", "--strict", "lesson_models.py"],
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout or result.stderr)
assert result.returncode == 0, result.stderr or result.stdout

Success: no issues found in 1 source file



## 9. Práctica guiada: de entradas crudas a reporte

Estos seis registros fueron escritos para esta lección y se pueden reutilizar con
los materiales del curso. Incluyen dos válidos, un score fuera de rango, una
etiqueta nula, un score ausente y una entrada que no es un diccionario.

Antes de ejecutar: identifica posiciones válidas y rechazadas. Esperamos **2
aceptadas**, **4 posiciones rechazadas**, promedio **0.49** y **1 seleccionada** con
umbral 0.9. No calculamos exactitud: no existen etiquetas de referencia.

In [67]:
raw_records: list[object] = [
    {"text": " Excellent ", "label": " POSITIVE ", "score": "0.98"},
    {"text": "Zero is valid", "label": "neutral", "score": 0.0},
    {"text": "Out of range", "label": "negative", "score": 1.2},
    {"text": "Missing label", "label": None, "score": 0.8},
    {"text": "Missing score", "label": "neutral"},
    None,
]

batch_config = models.BatchConfig(batch_name="Class demo", threshold=0.9, max_records=10)
report = process_records(raw_records, batch_config)
rejected_positions = {issue.position for issue in report.issues}
print(f"Accepted: {len(report.accepted)}")
print(f"Rejected records: {len(rejected_positions)}")
print(f"Average score: {average_score(report.accepted)}")
print(f"Selected: {len(select_predictions(report))}")
for issue in report.issues:
    print(issue.position, issue.field, issue.error_type)

Accepted: 2
Rejected records: 4
Average score: 0.49
Selected: 1
2 score less_than_equal
3 label literal_error
4 score missing
5 <record> model_type


### Ejercicio 7 · Comprobar antes de ampliar

Verifica los resultados esperados del lote. Procesa una lista vacía y otra con
solo `None`: sus promedios deben ser `None`. Prueba un lote mayor que el máximo:
debe producir `ValueError`, sin truncarlo silenciosamente. Usa el módulo de apoyo.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [68]:
# Write assertions for the demo batch and the boundary cases.

import math

# Mi predicción:
# El lote demo tiene 2 aceptadas, 4 rechazadas (posiciones 2, 3, 4 y 5),
# promedio 0.49 y 1 seleccionada con umbral 0.9.
# Una lista vacía y una lista con solo None no tienen aceptadas,
# así que su promedio debe ser None y no 0.
# Un lote con más registros que el máximo debe fallar con ValueError.

# Parte 1: compruebo los resultados del lote demo de la celda anterior.
assert len(report.accepted) == 2
assert rejected_positions == {2, 3, 4, 5}

promedio = average_score(report.accepted)
# Uso math.isclose porque es un decimal.
assert promedio is not None
assert math.isclose(promedio, 0.49)

assert len(select_predictions(report)) == 1
print("Lote demo: 2 aceptadas, posiciones rechazadas", sorted(rejected_positions), "promedio", promedio)

# Parte 2: lista vacía. No hay registros, así que no hay aceptadas ni problemas.
reporte_vacio = process_records([], batch_config)
assert reporte_vacio.accepted == []
assert reporte_vacio.issues == []
assert average_score(reporte_vacio.accepted) is None
print("Lista vacía -> promedio:", average_score(reporte_vacio.accepted))

# Parte 3: lista con solo None. El registro se rechaza porque no es un diccionario.
reporte_none = process_records([None], batch_config)
assert reporte_none.accepted == []
assert len(reporte_none.issues) == 1
assert average_score(reporte_none.accepted) is None
print("Solo None -> promedio:", average_score(reporte_none.accepted), "| problemas:", len(reporte_none.issues))

# Parte 4: un lote más grande que el máximo permitido.
# Creo una configuración con máximo 3 y le paso 4 registros.
config_chica = models.BatchConfig(batch_name="Lote chico", max_records=3)
lote_grande = [{"text": "Ok", "label": "neutral", "score": 0.5}] * 4

try:
    process_records(lote_grande, config_chica)
except ValueError as error:
    print("Lote de 4 con máximo 3:", error)
else:
    raise AssertionError("Un lote mayor al máximo debería producir ValueError")

# Explicación:
# Comprobé el lote demo antes de ampliar el módulo, para tener una referencia
# de que todo funciona bien antes de hacer cambios.
# Una lista vacía o con solo None da promedio None, porque no hay datos válidos:
# reportar 0 sería inventar un resultado.
# Un lote demasiado grande se rechaza completo con ValueError. Si se recortara
# en silencio, se perderían registros sin que nadie se diera cuenta.
# En mi trabajo sería como recibir más muestras de las que el laboratorio puede
# procesar: no se descartan las sobrantes sin avisar; se detiene y se reporta.

Lote demo: 2 aceptadas, posiciones rechazadas [2, 3, 4, 5] promedio 0.49
Lista vacía -> promedio: None
Solo None -> promedio: None | problemas: 1
Lote de 4 con máximo 3: Batch exceeds max_records


### Ejercicio 8 · Actividad independiente

Escribe una función tipada `count_labels(predictions)` que devuelva un diccionario
de conteos por etiqueta. Usa `list[models.Prediction]` como entrada y `dict[str, int]`
como salida. Debe funcionar para lista vacía. Añádela a una copia de tu módulo,
compruébala y ejecuta mypy sobre esa copia. Dentro de ese módulo usa
`list[Prediction]`, porque la clase se define allí; el prefijo `models` se usa
solo desde esta notebook.

Como parte de la misma evidencia, crea un registro con **dos errores**, conserva
ambos y explica por qué cuenta como **un registro rechazado**. Serializa el reporte
y comprueba que se pueda reconstruir.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [69]:
# Implement count_labels and check it with validated predictions.
# Add the function to your own module copy after trying it here.

import shutil
import subprocess
import sys

# Mi predicción:
# count_labels devuelve un diccionario con cuántas predicciones hay de cada etiqueta,
# y con una lista vacía devuelve un diccionario vacío.
# Un registro con dos errores genera dos problemas en el reporte,
# pero ambos tienen la misma posición, así que es un solo registro rechazado.

# Parte 1a: escribo y pruebo count_labels aquí en la libreta.
# Desde la libreta uso models.Prediction, porque importé el módulo como models.
def count_labels(predictions: list[models.Prediction]) -> dict[str, int]:
    '''Cuenta cuántas predicciones tienen cada etiqueta.'''
    conteos: dict[str, int] = {}
    for prediction in predictions:
        # Si la etiqueta no está todavía, get() devuelve 0; luego le sumo 1.
        conteos[prediction.label] = conteos.get(prediction.label, 0) + 1
    return conteos


validas = [
    models.Prediction(text="Excelente", label="positive", score=0.95),
    models.Prediction(text="Regular", label="neutral", score=0.6),
    models.Prediction(text="Muy bien", label="positive", score=0.8),
]
print("Conteo:", count_labels(validas))
assert count_labels(validas) == {"positive": 2, "neutral": 1}

# Con una lista vacía debe funcionar y devolver un diccionario vacío.
assert count_labels([]) == {}

# Parte 1b: copio el módulo con otro nombre y le agrego la función.
# Así no se pierde mi cambio si vuelvo a ejecutar la celda %%writefile.
shutil.copy("lesson_models.py", "lesson_models_pamela.py")

# Dentro del módulo uso Prediction sin el prefijo models,
# porque la clase está definida en ese mismo archivo.
funcion = '''

def count_labels(predictions: list[Prediction]) -> dict[str, int]:
    """Count how many predictions have each label."""
    counts: dict[str, int] = {}
    for prediction in predictions:
        counts[prediction.label] = counts.get(prediction.label, 0) + 1
    return counts
'''
# "a" significa agregar al final del archivo sin borrar lo que ya tiene.
with open("lesson_models_pamela.py", "a", encoding="utf-8") as archivo:
    archivo.write(funcion)

# Reviso mi copia del módulo con mypy en modo estricto.
resultado = subprocess.run(
    [sys.executable, "-m", "mypy", "--strict", "lesson_models_pamela.py"],
    capture_output=True,
    text=True,
    check=False,
)
print("mypy:", resultado.stdout.strip() or resultado.stderr.strip())
assert resultado.returncode == 0

# Importo mi copia y compruebo que la función también funciona desde ahí.
import lesson_models_pamela as mis_modelos
assert mis_modelos.count_labels([]) == {}

# Parte 2: un registro con dos errores.
# El segundo registro tiene texto vacío (solo espacios) Y una etiqueta inválida.
registros = [
    {"text": "Buen servicio", "label": "positive", "score": 0.9},
    {"text": "   ", "label": "desconocida", "score": 0.5},
]
config_e8 = mis_modelos.BatchConfig(batch_name="Ejercicio 8")
reporte = mis_modelos.process_records(registros, config_e8)

for issue in reporte.issues:
    print("Problema:", issue.position, issue.field, issue.error_type)

# Hay dos problemas guardados...
assert len(reporte.issues) == 2
# ...pero los dos son de la posición 1, así que es un solo registro rechazado.
assert {issue.position for issue in reporte.issues} == {1}
assert len(reporte.accepted) == 1
print("Conteo del reporte:", mis_modelos.count_labels(reporte.accepted))

# Parte 3: serializo el reporte a JSON, lo guardo en un archivo y lo reconstruyo.
json_reporte = reporte.model_dump_json(indent=2)
print(json_reporte)

with open("reporte_ejercicio8.json", "w", encoding="utf-8") as archivo:
    archivo.write(json_reporte)

reconstruido = mis_modelos.BatchReport.model_validate_json(json_reporte)
assert reconstruido == reporte
print("Reporte reconstruido igual al original:", reconstruido == reporte)

# Explicación:
# El registro de la posición 1 tiene dos errores: el texto queda vacío al quitar
# los espacios y la etiqueta no está permitida. Conservo los dos problemas porque
# cada uno dice algo distinto que hay que corregir, pero los dos pertenecen al
# mismo registro. Por eso hay 2 problemas y 1 solo registro rechazado.
# Contar problemas en lugar de registros inflaría el número de rechazos.
# En mi trabajo es como una muestra que llega con la etiqueta ilegible y el
# sello roto: son dos no conformidades, pero es una sola muestra rechazada.

Conteo: {'positive': 2, 'neutral': 1}
mypy: Success: no issues found in 1 source file
Problema: 1 text string_too_short
Problema: 1 label literal_error
Conteo del reporte: {'positive': 1}
{
  "config": {
    "batch_name": "Ejercicio 8",
    "threshold": 0.9,
    "max_records": 100
  },
  "accepted": [
    {
      "text": "buen servicio",
      "label": "positive",
      "score": 0.9,
      "source": null
    }
  ],
  "issues": [
    {
      "position": 1,
      "field": "text",
      "message": "String should have at least 1 character",
      "error_type": "string_too_short"
    },
    {
      "position": 1,
      "field": "label",
      "message": "Input should be 'positive', 'neutral' or 'negative'",
      "error_type": "literal_error"
    }
  ]
}
Reporte reconstruido igual al original: True


## 10. Una API pequeña con FastAPI y Pydantic

Una API puede recibir JSON cuyos valores no conocemos al escribir el programa.
FastAPI usa el modelo `QuoteRequest` para validar el cuerpo de la petición
antes de llamar a la función. Si los datos son válidos, la función recibe
una instancia con sus campos ya convertidos. `response_model` describe y
valida la respuesta.

`TestClient` enviará peticiones a la aplicación desde esta notebook, sin
iniciar un servidor. La organización de la aplicación como proyecto vendrá
en la sesión de uv.

In [70]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field


class QuoteRequest(BaseModel):
    item_name: str = Field(min_length=1)
    unit_price: float = Field(gt=0)
    quantity: int = Field(ge=1)


class QuoteResponse(BaseModel):
    item_name: str
    unit_price: float
    quantity: int
    total: float


app = FastAPI()


@app.post("/quotes", response_model=QuoteResponse)
def create_quote(quote: QuoteRequest) -> QuoteResponse:
    return QuoteResponse(
        item_name=quote.item_name,
        unit_price=quote.unit_price,
        quantity=quote.quantity,
        total=quote.unit_price * quote.quantity,
    )


client = TestClient(app)

### Una petición válida y otra que requiere conversión

En la segunda petición, `unit_price` llega como texto. Pydantic comprueba
que pueda convertirse a `float`; la función de la ruta recibe el número y
calcula el total. La respuesta JSON contiene un número, no el texto original.

In [71]:
valid_response = client.post(
    "/quotes",
    json={"item_name": "Notebook", "unit_price": 19.5, "quantity": 2},
)
converted_response = client.post(
    "/quotes",
    json={"item_name": "Notebook", "unit_price": "19.5", "quantity": 2},
)

assert valid_response.status_code == 200
assert converted_response.status_code == 200
assert valid_response.json() == converted_response.json()
assert converted_response.json()["total"] == 39.0
print(converted_response.json())

{'item_name': 'Notebook', 'unit_price': 19.5, 'quantity': 2, 'total': 39.0}


### Una petición rechazada

El texto `"nineteen"` no puede convertirse a `float` y una cantidad de
cero incumple la restricción `ge=1`. FastAPI responde con estado HTTP 422
y señala ambos campos. La función `create_quote` no se ejecuta para esta
petición.

In [72]:
invalid_response = client.post(
    "/quotes",
    json={"item_name": "Notebook", "unit_price": "nineteen", "quantity": 0},
)

assert invalid_response.status_code == 422
issues = invalid_response.json()["detail"]
assert {tuple(issue["loc"]) for issue in issues} == {
    ("body", "unit_price"),
    ("body", "quantity"),
}
for issue in issues:
    print(issue["loc"], issue["type"])

['body', 'unit_price'] float_parsing
['body', 'quantity'] greater_than_equal


## Evidencia y criterios

Conserva tu módulo con la función del ejercicio 8,
las comprobaciones y un reporte JSON. Puedes copiar la salida JSON o guardarla
localmente; todavía no se exige una base de datos ni un proyecto empaquetado.

- **Contratos claros:** parámetros y retornos anotados; ausencia de `Any` injustificado.
- **Validación:** restricciones y normalización justificadas; cero válido, faltantes y
  entradas inválidas distinguidos; configuración validada antes de procesar.
- **Errores útiles:** posición, campo y motivo conservados, incluidos varios errores por registro.
- **Serialización:** reconstrucción equivalente del reporte anidado.
- **Verificación:** mypy sin errores en el módulo final y comprobaciones de casos límite.

### Profundización

Compara `model_dump(exclude_none=True)` con la salida habitual. ¿Cuál es la diferencia
entre omitir `source` en la representación y cambiar el contrato del modelo?

## Cierre

Explica por qué una anotación no reemplaza la validación, qué información aporta
un error de Pydantic y qué decisiones de conversión tomaste. La próxima sesión
procesa datos de forma perezosa con iteradores, generadores y context managers.

## Referencias

- [Pydantic: modelos](https://docs.pydantic.dev/latest/concepts/models/).
- [Pydantic: validadores](https://docs.pydantic.dev/latest/concepts/validators/).
- [mypy: primeros pasos](https://mypy.readthedocs.io/en/stable/getting_started.html).
- [FastAPI: cuerpo de petición](https://fastapi.tiangolo.com/tutorial/body/).
- [FastAPI: TestClient](https://fastapi.tiangolo.com/tutorial/testing/).

- [PEP 1: propósito y proceso](https://peps.python.org/pep-0001/).
- [PEP 484: Type Hints](https://peps.python.org/pep-0484/).
